# Data Collection (ml.m5.12xlarge)

In [1]:
import boto3
import pandas as pd
import numpy as np
import os
import pickle
from pandas.api.types import is_numeric_dtype
from pprint import pprint

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Define functions

In [2]:
# import and concatenate
def import_from_filename_list(list_str_files, int_n_rows=None):
    list_df = []
    for a, str_file in enumerate(list_str_files):
        # read
        df = pd.read_csv(f's3://{str_project}/{str_file}', sep='|', nrows=int_n_rows)
        # filename
        try:
            str_filename = str_file.split('/')[3]
        except IndexError:
            str_filename = str_file.split('/')[2]
        # shape
        tpl_shape = df.shape
        # print info
        print(f'{a+1}/{len(list_str_files)} - {str_filename} - N rows: {tpl_shape[0]} - N columns: {tpl_shape[1]}')
        # append
        list_df.append(df)
    return list_df

In [3]:
# get info
def get_df_info(df):
    # get info
    tpl_shape = df.shape
    print(f'N rows: {tpl_shape[0]}; N columns: {tpl_shape[1]}')

In [4]:
# show columns
def show_columns(df):
    for col in df.columns:
        print(col)

In [5]:
# get common and uncommon items in 2 lists
def get_common_and_uncommon_columns(list_a, list_b):
    list_common = []
    list_uncommon = []
    for col in list_a:
        if col in list_b:
            list_common.append(col)
        else:
            list_uncommon.append(col)
    # opposite
    for col in list_b:
        if col in list_a:
            list_common.append(col)
        else:
            list_uncommon.append(col)
    # rm dups
    list_common = list(dict.fromkeys(list_common))
    list_uncommon = list(dict.fromkeys(list_uncommon))
    # return
    return list_common, list_uncommon

In [6]:
# create target
def create_target(str_performance, list_str_perf=['30', '60', '90+', 'CO', 'BK']):
    if str_performance in list_str_perf:
        return 1
    else:
        return 0

## Constants

In [7]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# output
str_dirname_output = './output'
# n rows (for testing)
int_n_rows = None
#int_n_rows = 1000
# bucket
cls_bucket = boto3.resource('s3').Bucket(str_project)

Project: 20231010-gen-xii


## Create output directory

In [8]:
# create dir
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    print(f'Directory {str_dirname_output} already exists')

Directory ./output already exists


## Get performance file as base table

In [9]:
# import
str_filename = 'PRM.EDTOUT.DGMPRSTG.1273669.PERF.CSV'
str_uri = f's3://{str_project}/01_ad/01_data_prep/01_data_collection/input/match-file/return-data/{str_filename}'
df_base = pd.read_csv(
    str_uri,
    nrows=int_n_rows,
)
# get info
get_df_info(df=df_base)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:272: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


N rows: 958287; N columns: 29


In [10]:
#pprint(list(df_base.columns))

In [11]:
# check proportion NaN in performance
flt_prop_nan = df_base['performance'].isnull().mean()
print(f'Proportion NaN in performance column: {flt_prop_nan}')

Proportion NaN in performance column: 0.546805915138158


In [12]:
#  drop all rows where performance is nan
df_base.dropna(subset=['performance'], inplace=True)
# get info
get_df_info(df=df_base)

N rows: 434290; N columns: 29


In [13]:
# check proportion NaN in performance
flt_prop_nan = df_base['performance'].isnull().mean()
print(f'Proportion NaN in performance column: {flt_prop_nan}')

Proportion NaN in performance column: 0.0


In [14]:
# freq
df_base['performance'].value_counts()

performance
Current    253190
CO          68774
30          57064
60          26732
90+         19208
BK           9322
Name: count, dtype: int64

In [15]:
# append suffix
df_base.columns = [f'{col}__base' for col in df_base.columns]

In [16]:
#pprint(list(df_base.columns))

In [17]:
# make sure no rows are NaN in ID and target cols
df_base[['uniqueid__base', 'performance__base']].isnull().mean()

uniqueid__base       0.0
performance__base    0.0
dtype: float64

In [18]:
# make sure there are no duplicate ids
print(f'There are {df_base["uniqueid__base"].duplicated().sum()} duplicate IDs')

There are 0 duplicate IDs


### Important: ```uniqueid__base``` is the column used for joining to the match file

## Join the accept (received) tables from TU

In [19]:
# get accept files
str_prefix = '01_ad/01_data_prep/01_data_collection/input/match-file/return-data/'
list_str_files = []
for str_object in cls_bucket.objects.filter(Prefix=str_prefix):
    # create str
    str_file = str_object.key
    # logic
    if 'ACC2' in str_file:
        # print
        print(str_file)
        # append
        list_str_files.append(str_file)

# import
list_df = import_from_filename_list(
    list_str_files=list_str_files,
    int_n_rows=int_n_rows,
)

# concatenate vertically
df_received = pd.concat(list_df, axis=0)
# get info
get_df_info(df=df_received)

01_ad/01_data_prep/01_data_collection/input/match-file/return-data/PRM.EDTOUT.DGMPRSTG.P467632.20180930-ACC2-P001.CSV
01_ad/01_data_prep/01_data_collection/input/match-file/return-data/PRM.EDTOUT.DGMPRSTG.P467633.20181231-ACC2-P001.CSV
01_ad/01_data_prep/01_data_collection/input/match-file/return-data/PRM.EDTOUT.DGMPRSTG.P467634.20190331-ACC2-P001.CSV
01_ad/01_data_prep/01_data_collection/input/match-file/return-data/PRM.EDTOUT.DGMPRSTG.P467635.20190630-ACC2-P001.CSV
01_ad/01_data_prep/01_data_collection/input/match-file/return-data/PRM.EDTOUT.DGMPRSTG.P467636.20190930-ACC2-P001.CSV


/tmp/ipykernel_32684/1623506151.py:6: DtypeWarning: Columns (728,729,730,731,732) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f's3://{str_project}/{str_file}', sep='|', nrows=int_n_rows)


1/5 - input - N rows: 193964 - N columns: 1846


/tmp/ipykernel_32684/1623506151.py:6: DtypeWarning: Columns (726,730,731,732) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f's3://{str_project}/{str_file}', sep='|', nrows=int_n_rows)


2/5 - input - N rows: 208297 - N columns: 1846


...

3/5 - input - N rows: 196990 - N columns: 1846


/tmp/ipykernel_32684/1623506151.py:6: DtypeWarning: Columns (725,726,728,729,730,731,732) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f's3://{str_project}/{str_file}', sep='|', nrows=int_n_rows)


4/5 - input - N rows: 198963 - N columns: 1846


/tmp/ipykernel_32684/1623506151.py:6: DtypeWarning: Columns (720,721,722,723,724,725,726,727,728,729,730,731,732) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f's3://{str_project}/{str_file}', sep='|', nrows=int_n_rows)


5/5 - input - N rows: 173796 - N columns: 1846
N rows: 972010; N columns: 1846


In [20]:
#pprint(list(df_received.columns))

In [21]:
# empty list
list_cols = ['permId']
for col in df_received.columns:
    if 'score' in col:
        list_cols.append(col)
# show
pprint(list_cols)

# drop these
df_received.drop(list_cols, axis=1, inplace=True)

['permId',
 'eads162_score',
 'eads202_score',
 'eads232_score',
 'eaup02_score',
 'eoap02_score',
 'aadm42_score',
 'aadm43_score',
 'aadm44_score',
 'aadm45_score',
 'aadm143_score',
 'aadm144_score',
 'eads19_score',
 'eads51_score',
 'etie04_score',
 'edie04_score',
 'evtg04_finscore',
 'cvtg03_finscore']


In [22]:
# get info
get_df_info(df=df_received)

N rows: 972010; N columns: 1828


In [23]:
# get new col names
list_cols = []
for col in df_received.columns:
    list_cols.append(f'{col.split("_")[1]}__tu')

# show it
#pprint(list_cols)

In [24]:
# assign
df_received.columns = list_cols

In [25]:
#pprint(list(df_received.columns))

In [26]:
# join
df_base = pd.merge(
    left=df_base,
    right=df_received,
    left_on='uniqueid__base',
    right_on='uniqueid__tu',
    how='left',
)

# save memory
del df_received

# get info
get_df_info(df=df_base)

N rows: 434290; N columns: 1857


In [27]:
# list cols to drop
list_cols = [
    'uniqueid__tu',
    'bigaccountid__tu',
    'bigdebtorid__tu',
]

# drop
df_base.drop(list_cols, axis=1, inplace=True)

# get info
get_df_info(df=df_base)

N rows: 434290; N columns: 1854


In [28]:
# make sure no rows are NaN in ID and target cols
df_base[['uniqueid__base', 'performance__base']].isnull().mean()

uniqueid__base       0.0
performance__base    0.0
dtype: float64

In [29]:
# make sure there are no duplicate ids
print(f'There are {df_base["uniqueid__base"].duplicated().sum()} duplicate IDs')

There are 0 duplicate IDs


### Match file (to get UniqueID, bigAccountId, and bigDebtorId from TU)

In [30]:
# import
str_filename = 'match_file.zip'
str_uri = f's3://{str_project}/01_ad/01_data_prep/01_data_collection/input/match-file/{str_filename}'
df_match = pd.read_csv(
    str_uri,
    usecols=['UniqueID', 'bigAccountId', 'bigDebtorId', 'UniqueID_TU'],
    #nrows=int_n_rows,
)

# get info
get_df_info(df=df_match)

N rows: 972292; N columns: 4


### Important: ```UniqueID_TU``` is the column in the match file to use for joining to the base TU data on ```uniqueid__base```

In [31]:
# preview data
df_match.head(5)

,UniqueID,bigAccountId,bigDebtorId,UniqueID_TU
0,390847750287411,3908477,5028741,524304540300867
1,390847750287420,3908477,5028742,524304540300876
2,390933050297991,3909330,5029799,524490840310347
3,390933050298000,3909330,5029800,524490840311456
4,390844350286981,3908443,5028698,524301140309337


In [32]:
# join
df_base = pd.merge(
    left=df_base,
    right=df_match,
    left_on='uniqueid__base',
    right_on='UniqueID_TU',
    how='left',
)

# save memory
del df_match

In [33]:
#pprint(list(df_base.columns))

In [34]:
# list cols to drop
list_cols = [
    'uniqueid__base',
    'bigaccountid__base',
    'bigdebtorid__base',
    'UniqueID_TU',
]

# drop 
df_base.drop(list_cols, axis=1, inplace=True)

# get info
get_df_info(df=df_base)

N rows: 434290; N columns: 1854


### Important: after dropping the features above, ```UniqueID``` is the key for joining the files sent to TU

In [35]:
#pprint(list(df_base.columns))

In [36]:
# make sure no rows are NaN in ID and target cols
df_base[['UniqueID', 'performance__base']].isnull().mean()

UniqueID             0.0
performance__base    0.0
dtype: float64

In [37]:
# make sure there are no duplicate ids
print(f'There are {df_base["UniqueID"].duplicated().sum()} duplicate IDs')

There are 0 duplicate IDs


### Important: Now, can we join ```df_received``` to the tables sent to TU on ```UniqueID```

## Join the sent files (to TU)

In [38]:
# get files
str_prefix = '01_ad/01_data_prep/01_data_collection/input/match-file/sent-data/'
list_str_files = []
for str_object in cls_bucket.objects.filter(Prefix=str_prefix):
    # create str
    str_file = str_object.key
    # logic
    if ('.csv' in str_file) and ('Debt' not in str_file) and ('TU' not in str_file):
        # print
        print(str_file)
        # append
        list_str_files.append(str_file)

01_ad/01_data_prep/01_data_collection/input/match-file/sent-data/Application_10012018_01012020.csv
01_ad/01_data_prep/01_data_collection/input/match-file/sent-data/Income_10012018_01012020.csv
01_ad/01_data_prep/01_data_collection/input/match-file/sent-data/LN_10012018_01012020.csv


### Application

In [39]:
# read
str_filename = 'Application_10012018_01012020.csv'
str_uri = f's3://{str_project}/01_ad/01_data_prep/01_data_collection/input/match-file/sent-data/{str_filename}'
df_app = pd.read_csv(
    str_uri,
    #nrows=int_n_rows,
)

# get info
get_df_info(df=df_app)

/tmp/ipykernel_32684/1376576636.py:4: DtypeWarning: Columns (6,30) have mixed types. Specify dtype option on import or set low_memory=False.
  df_app = pd.read_csv(


N rows: 972297; N columns: 80


In [40]:
#pprint(list(df_app.columns))

In [41]:
# suffix
df_app.columns = [f'{col}__app' for col in df_app.columns]

In [42]:
#pprint(list(df_app.columns))

In [43]:
# preview date col
df_app['ApplicationDate__app'].head(5)

0    20181001
1    20181001
2    20181001
3    20181001
4    20181001
Name: ApplicationDate__app, dtype: int64

In [44]:
# convert applicationdate__app to dt
df_app['ApplicationDate__app'] = pd.to_datetime(df_app['ApplicationDate__app'], format='%Y%m%d')

# preview date col
df_app['ApplicationDate__app'].head(5)

0   2018-10-01
1   2018-10-01
2   2018-10-01
3   2018-10-01
4   2018-10-01
Name: ApplicationDate__app, dtype: datetime64[ns]

In [45]:
# join
df_base = pd.merge(
    left=df_base,
    right=df_app,
    left_on='UniqueID',
    right_on='UniqueID__app',
    how='left',
)

# save memory
del df_app

# get info
get_df_info(df=df_base)

N rows: 434290; N columns: 1934


In [46]:
# cols to drop
list_cols = [
    'UniqueID__app',
    'bigAccountId__app',
    'bigDebtorId__app',
]

# drop
df_base.drop(list_cols, axis=1, inplace=True)

# get info
get_df_info(df=df_base)

N rows: 434290; N columns: 1931


In [47]:
# preview date col
df_base['ApplicationDate__app'].head(5)

0   2018-11-03
1   2018-11-03
2   2018-11-03
3   2018-10-30
4   2018-10-31
Name: ApplicationDate__app, dtype: datetime64[ns]

In [48]:
# make sure no rows are NaN in ID, date, and target cols
df_base[['UniqueID', 'ApplicationDate__app', 'performance__base']].isnull().mean()

UniqueID                0.0
ApplicationDate__app    0.0
performance__base       0.0
dtype: float64

In [49]:
# make sure there are no duplicate ids
print(f'There are {df_base["UniqueID"].duplicated().sum()} duplicate IDs')

There are 0 duplicate IDs


### Lexus Nexus

In [50]:
# read
str_filename = 'LN_10012018_01012020.csv'
str_uri = f's3://{str_project}/01_ad/01_data_prep/01_data_collection/input/match-file/sent-data/{str_filename}'
df_ln = pd.read_csv(
    str_uri,
    nrows=int_n_rows,
)

# get info
get_df_info(df=df_ln)

N rows: 972297; N columns: 193


In [51]:
#pprint(list(df_ln.columns))

In [52]:
# suffix
df_ln.columns = [f'{col}__ln' for col in df_ln.columns]

In [53]:
#pprint(list(df_ln.columns))

In [54]:
# join
df_base = pd.merge(
    left=df_base,
    right=df_ln,
    left_on='UniqueID',
    right_on='UniqueID__ln',
    how='left',
)

# save memory
del df_ln

# get info
get_df_info(df=df_base)

N rows: 434290; N columns: 2124


In [55]:
#pprint(list(df_base.columns))

In [56]:
# cols to drop
list_cols = [
    'UniqueID__ln',
    'bigAccountId__ln',
    'bigDebtorId__ln',
]

# drop
df_base.drop(list_cols, axis=1, inplace=True)

# get info
get_df_info(df=df_base)

N rows: 434290; N columns: 2121


In [57]:
# preview date col
df_base['ApplicationDate__app'].head(5)

0   2018-11-03
1   2018-11-03
2   2018-11-03
3   2018-10-30
4   2018-10-31
Name: ApplicationDate__app, dtype: datetime64[ns]

In [58]:
# make sure no rows are NaN in ID, date, and target cols
df_base[['UniqueID', 'ApplicationDate__app', 'performance__base']].isnull().mean()

UniqueID                0.0
ApplicationDate__app    0.0
performance__base       0.0
dtype: float64

In [59]:
# make sure there are no duplicate ids
print(f'There are {df_base["UniqueID"].duplicated().sum()} duplicate IDs')

There are 0 duplicate IDs


### Income

In [60]:
# read
str_filename = 'Income_10012018_01012020.csv'
str_uri = f's3://{str_project}/01_ad/01_data_prep/01_data_collection/input/match-file/sent-data/{str_filename}'
df_income = pd.read_csv(
    str_uri,
    usecols=['UniqueID', 'bitInvalid', 'bitUse', 'fltGrossMonthly'],
    nrows=int_n_rows,
)
df_income['bitInvalid'] = df_income['bitInvalid'].astype(float)
df_income['bitUse'] = df_income['bitUse'].astype(float)

# get info
get_df_info(df=df_income)

N rows: 1472267; N columns: 4


In [61]:
# filter out rows we dont want
df_income = df_income[df_income['bitInvalid']==0.0]
df_income = df_income[df_income['bitUse']==1.0]

# drop
df_income.drop(['bitInvalid', 'bitUse'], axis=1, inplace=True)

# get info
get_df_info(df=df_income)

N rows: 1414409; N columns: 2


In [62]:
# suffix
df_income.columns = [f'{col}__income' for col in df_income.columns]

In [63]:
pprint(list(df_income.columns))

['UniqueID__income', 'fltGrossMonthly__income']


In [64]:
# agg dict
dict_agg = {
    'fltGrossMonthly__income': ['sum', 'count'],
}

# save
pickle.dump(dict_agg, open(f'{str_dirname_output}/dict_agg.pkl', 'wb'))

In [65]:
# aggregate
df_income = df_income.groupby('UniqueID__income', as_index=False).agg(dict_agg)

# get info
get_df_info(df=df_income)

N rows: 956297; N columns: 3


In [66]:
df_income.head(5)

UniqueID__income fltGrossMonthly__income      
                                       sum count
0  390763350277011                  7200.0     2
1  390763450277021                  5052.0     1
2  390763550277031                  3400.0     1
3  390763550277040                  4000.0     2
4  390763750277061                  2890.0     2

In [67]:
# assign cols
df_income.columns = ['UniqueID__income', 'fltGrossMonthly__income_sum', 'fltGrossMonthly__income_count']

# preview
df_income.head(5)

,UniqueID__income,fltGrossMonthly__income_sum,fltGrossMonthly__income_count
0,390763350277011,7200.0,2
1,390763450277021,5052.0,1
2,390763550277031,3400.0,1
3,390763550277040,4000.0,2
4,390763750277061,2890.0,2


In [68]:
# join
df_base = pd.merge(
    left=df_base,
    right=df_income,
    left_on='UniqueID',
    right_on='UniqueID__income',
    how='left',
)

# save memory
del df_income

# get info
get_df_info(df=df_base)

N rows: 434290; N columns: 2124


In [69]:
#pprint(list(df_base.columns))

In [70]:
# cols to drop
list_cols = [
    'UniqueID__income',
]

# drop
df_base.drop(list_cols, axis=1, inplace=True)

# get info
get_df_info(df=df_base)

N rows: 434290; N columns: 2123


In [71]:
# preview date col
df_base['ApplicationDate__app'].head(5)

0   2018-11-03
1   2018-11-03
2   2018-11-03
3   2018-10-30
4   2018-10-31
Name: ApplicationDate__app, dtype: datetime64[ns]

In [72]:
# make sure no rows are NaN in ID, date, and target cols
df_base[['UniqueID', 'ApplicationDate__app', 'performance__base']].isnull().mean()

UniqueID                0.0
ApplicationDate__app    0.0
performance__base       0.0
dtype: float64

In [73]:
# make sure there are no duplicate ids
print(f'There are {df_base["UniqueID"].duplicated().sum()} duplicate IDs')

There are 0 duplicate IDs


## Append the population proxy (received) files from TU

In [74]:
# get population proxy files
str_prefix = '01_ad/01_data_prep/01_data_collection/input/match-file/proxy-population/'
list_str_files = []
for str_object in cls_bucket.objects.filter(Prefix=str_prefix):
    # create str
    str_file = str_object.key
    # logic
    if '.CSV' in str_file:
        # print
        print(str_file)
        # append
        list_str_files.append(str_file)

# import
list_df = import_from_filename_list(
    list_str_files=list_str_files,
    int_n_rows=int_n_rows,
)

# concatenate vertically
df_proxy = pd.concat(list_df, axis=0)

# get info
get_df_info(df=df_proxy)

01_ad/01_data_prep/01_data_collection/input/match-file/proxy-population/PRM.EDTOUT.DGMPRSTG.P472176.20180930.ACC1.P001.CSV
01_ad/01_data_prep/01_data_collection/input/match-file/proxy-population/PRM.EDTOUT.DGMPRSTG.P472177.20181231.ACC1.P001.CSV
01_ad/01_data_prep/01_data_collection/input/match-file/proxy-population/PRM.EDTOUT.DGMPRSTG.P472178.20190331.ACC1.P001.CSV
01_ad/01_data_prep/01_data_collection/input/match-file/proxy-population/PRM.EDTOUT.DGMPRSTG.P472179.20190630.ACC1.P001.CSV
01_ad/01_data_prep/01_data_collection/input/match-file/proxy-population/PRM.EDTOUT.DGMPRSTG.P472180.20190930.ACC1.P001.CSV
1/5 - input - N rows: 26318 - N columns: 1853


/tmp/ipykernel_32684/1623506151.py:6: DtypeWarning: Columns (1843) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f's3://{str_project}/{str_file}', sep='|', nrows=int_n_rows)


2/5 - input - N rows: 29800 - N columns: 1853


/tmp/ipykernel_32684/1623506151.py:6: DtypeWarning: Columns (1842) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f's3://{str_project}/{str_file}', sep='|', nrows=int_n_rows)


3/5 - input - N rows: 30325 - N columns: 1853


/tmp/ipykernel_32684/1623506151.py:6: DtypeWarning: Columns (1842) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f's3://{str_project}/{str_file}', sep='|', nrows=int_n_rows)


4/5 - input - N rows: 29994 - N columns: 1853
5/5 - input - N rows: 27306 - N columns: 1853
N rows: 143743; N columns: 1853


In [75]:
# preview date col
df_proxy['customerInput_openDte'].head(5)

0    2018-10-22
1    2018-12-12
2    2018-11-08
3    2018-10-23
4    2018-12-11
Name: customerInput_openDte, dtype: object

In [76]:
# convert to datetime
df_proxy['customerInput_openDte'] = pd.to_datetime(df_proxy['customerInput_openDte'])

# preview date col
df_proxy['customerInput_openDte'].head(5)

0   2018-10-22
1   2018-12-12
2   2018-11-08
3   2018-10-23
4   2018-12-11
Name: customerInput_openDte, dtype: datetime64[ns]

In [77]:
#pprint(list(df_proxy.columns))

In [78]:
# empty list
list_cols = []
for col in df_proxy.columns:
    if 'score' in col:
        list_cols.append(col)
# show
pprint(list_cols)

# drop these
df_proxy.drop(list_cols, axis=1, inplace=True)

# get info
get_df_info(df=df_proxy)

['aadm42_score',
 'aadm43_score',
 'aadm44_score',
 'aadm45_score',
 'aadm143_score',
 'aadm144_score',
 'eads162_score',
 'eads202_score',
 'eads232_score',
 'eaup02_score',
 'eoap02_score',
 'eads19_score',
 'eads51_score',
 'cvtg03_finscore',
 'evtg04_finscore',
 'edie04_score',
 'etie04_score']
N rows: 143743; N columns: 1836


In [79]:
# empty list
list_cols = []
for col in df_proxy.columns:
    if 'advrs' in col:
        list_cols.append(col)
# show
pprint(list_cols)

# drop these
df_proxy.drop(list_cols, axis=1, inplace=True)

# get info
get_df_info(df=df_proxy)

['cvtg03_advrs1',
 'cvtg03_advrs2',
 'cvtg03_advrs3',
 'cvtg03_advrs4',
 'cvtg03_advrs5',
 'evtg04_advrs1',
 'evtg04_advrs2',
 'evtg04_advrs3',
 'evtg04_advrs4',
 'evtg04_advrs5']
N rows: 143743; N columns: 1826


In [80]:
# get new col names
list_cols = []
for col in df_proxy.columns:
    list_cols.append(f'{col.split("_")[1]}__tu')

# show it
#pprint(list_cols)

In [81]:
# assign
df_proxy.columns = list_cols

In [82]:
#pprint(list(df_proxy.columns))

In [83]:
# rename cols
dict_rename = {
    'openDte__tu': 'ApplicationDate__app',
    'highCrAmt__tu': 'high_cr_amt__base',
    'paytSchdMthCnt__tu': 'payt_schd_mth_cnt__base',
    'paytDueAmt__tu': 'payt_due_amt__base',
    'performance__tu': 'performance__base',
    
    'acctTypCde__tu': 'acct_typ_cde__base',
    'currBalAmt__tu': 'curr_bal_amt__base',
    'termsFreqCde__tu': 'terms_freq_cde__base',
    'affilRemCde__tu': 'affil_rem_cde__base',
    'cmplncRemCde__tu': 'cmplnc_rem_cde__base',
    'gnrcRemCde__tu': 'gnrc_rem_cde__base',
    'rteRemCde__tu': 'rte_rem_cde__base',
    'acctRtePrflCde__tu': 'acct_rte_prfl_cde__base',
    'paytPttrnTxt__tu': 'payt_pttrn_txt__base',
    'gnrcremFrsrptDte__tu': 'gnrcrem_frsrpt_dte__base',
    'obsArch__tu': 'obs_arch__base',
    'vtg4__tu': 'vtg4__base',
}

# rename
df_proxy.rename(columns=dict_rename, inplace=True)

In [84]:
#pprint(list(df_proxy.columns))

In [85]:
# concatenate
df_base = pd.concat([df_base, df_proxy], axis=0)

# save memory
del df_proxy

# get info
get_df_info(df=df_base)

N rows: 578033; N columns: 2127


In [86]:
# make sure no rows are NaN in ID, date, and target cols
df_base[['UniqueID', 'ApplicationDate__app', 'performance__base']].isnull().mean()

UniqueID                0.248676
ApplicationDate__app    0.000000
performance__base       0.000000
dtype: float64

In [87]:
# make sure there are no duplicate ids
print(f'There are {df_base["UniqueID"].dropna().duplicated().sum()} duplicate IDs')

There are 0 duplicate IDs


## Replace amount financed, term, and payment

In [88]:
# replace fltAmountFinanced__app with high_cr_amt__base and then drop high_cr_amt__base
df_base['fltAmountFinanced__app'] = df_base['high_cr_amt__base']
# drop
df_base.drop('high_cr_amt__base', axis=1, inplace=True)

# replace intTerm__app with payt_schd_mth_cnt__base and then drop payt_schd_mth_cnt__base
df_base['intTerm__app'] = df_base['payt_schd_mth_cnt__base']
# drop
df_base.drop('payt_schd_mth_cnt__base', axis=1, inplace=True)

# replace fltApprovedPayment__app with payt_due_amt__base and then drop payt_due_amt__base
df_base['fltApprovedPayment__app'] = df_base['payt_due_amt__base']
# drop
df_base.drop('payt_due_amt__base', axis=1, inplace=True)

# get info
get_df_info(df=df_base)

N rows: 578033; N columns: 2124


## Create target

In [89]:
# create 30+ dpd target
df_base['target'] = df_base['performance__base'].apply(
    lambda x: create_target(
        str_performance=x, 
        list_str_perf=['30','60','90+','CO','BK'],
    )
)
# freq
df_base['target'].value_counts()

/tmp/ipykernel_32684/1132714190.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_base['target'] = df_base['performance__base'].apply(


target
0    364822
1    213211
Name: count, dtype: int64

In [90]:
# # create 60+ dpd target
# df_base['target'] = df_base['performance__base'].apply(
#     lambda x: create_target(
#         str_performance=x, 
#         list_str_perf=['60','90+','CO','BK'],
#     )
# )
# # freq
# df_base['target'].value_counts()

In [91]:
# get info
get_df_info(df=df_base)

N rows: 578033; N columns: 2125


## Lower the columns

In [92]:
# lower col names
df_base.columns = [col.lower() for col in df_base.columns]

In [93]:
#pprint(list(df_base.columns))

In [94]:
# make sure no rows are NaN in ID, date, and target cols
df_base[['uniqueid', 'applicationdate__app', 'target']].isnull().mean()

uniqueid                0.248676
applicationdate__app    0.000000
target                  0.000000
dtype: float64

Note: ```NaN``` in ```unique_id``` are from population proxy

## Gen XI

In [95]:
# get column names from gen xi data
# Note: this is the same as: s3://20230131-cfpb-data-request/03_gen_xi/02_pd/01_data_collection/output/df_raw_sample.csv
str_filename = 'df_raw_sample.csv'
str_uri = f's3://{str_project}/01_ad/01_data_prep/01_data_collection/input/gen_xi/{str_filename}'
list_cols = list(pd.read_csv(str_uri).columns)
# replace ApplicationDate with ApplicationDate__app
list_cols = list(map(lambda x: x.replace('ApplicationDate', 'ApplicationDate__app'), list_cols))
# replace DPD60PLUS with target
list_cols = list(map(lambda x: x.replace('DPD60PLUS', 'target'), list_cols))
print(f'There are {len(list_cols)} columns in the Gen XI data:')
#pprint(list_cols)

/tmp/ipykernel_32684/1541505927.py:5: DtypeWarning: Columns (8,18,19,31,70,98,112,303,1650,1651,1652,1653,1654,1655,2091) have mixed types. Specify dtype option on import or set low_memory=False.
  list_cols = list(pd.read_csv(str_uri).columns)


There are 2421 columns in the Gen XI data:


In [96]:
# change tu suffix to match with the new data and also lower the gen xi columns because the gen xii columns are already lowered
list_cols_match = []
for col in list_cols:
    if '__tu' in col:
        list_cols_match.append(f"{col.split('__')[0]}__tu".lower())
    else:
        list_cols_match.append(col.lower())
#pprint(list_cols_match)

In [97]:
# create matching dict
dict_match = dict(zip(list_cols_match, list_cols))
#pprint(dict_match)

In [98]:
# get the columns we will use and those we wont
list_usecols = []
list_bad_cols = []
# look in the new columns
for col in list_cols_match:
    # get the corresponding old col
    str_col_old = dict_match[col]
    # if the column is in the new df
    if col in list(df_base.columns):
        # get the corresponding old column
        list_usecols.append(str_col_old)
    else:
        list_bad_cols.append(str_col_old)
# print
print(f'{len(list_usecols)} columns will be used, while {len(list_bad_cols)} columns will be disregarded:')

2058 columns will be used, while 363 columns will be disregarded:


In [99]:
pprint(list_bad_cols)

['bigAccountId__app',
 'bigDebtorId__app',
 'fltGrossMonthly__income_min',
 'fltGrossMonthly__income_max',
 'fltGrossMonthly__income_median',
 'fltGrossMonthly__income_mean',
 'fltGrossMonthly__income_std',
 'bigAccountId__ln',
 'bigDebtorId__ln',
 'ANALYTICSMATCHKEY__ln',
 'auto_score__ln',
 'auto_reason1__ln',
 'auto_reason2__ln',
 'auto_reason3__ln',
 'auto_reason4__ln',
 'auto_reason5__ln',
 'bankcard_score__ln',
 'bankcard_reason1__ln',
 'bankcard_reason2__ln',
 'bankcard_reason3__ln',
 'bankcard_reason4__ln',
 'bankcard_reason5__ln',
 'short_term_lending_score__ln',
 'short_term_lending_reason1__ln',
 'short_term_lending_reason2__ln',
 'short_term_lending_reason3__ln',
 'short_term_lending_reason4__ln',
 'short_term_lending_reason5__ln',
 'telecommunications_score__ln',
 'telecommunications_reason1__ln',
 'telecommunications_reason2__ln',
 'telecommunications_reason3__ln',
 'telecommunications_reason4__ln',
 'telecommunications_reason5__ln',
 'crossindustry_index__ln',
 'crossind

In [100]:
# show cols we will keep
#pprint(list_usecols)

In [101]:
%%time

# read from parquet
# Note: this is the same as: s3://20230131-cfpb-data-request/03_gen_xi/02_pd/01_data_collection/output/df_raw.gzip
str_filename = 'df_raw.gzip'
str_uri = f's3://{str_project}/01_ad/01_data_prep/01_data_collection/input/gen_xi/{str_filename}'
df_gen_xi = pd.read_parquet(str_uri)
df_gen_xi.replace(['NaN','nan',''], np.nan, inplace=True)
# rename ApplicationDate to ApplicationDate__app and DPD60PLUS to target
dict_rename = {
    'ApplicationDate': 'ApplicationDate__app',
    'DPD60PLUS': 'target',
}
# rename
df_gen_xi.rename(columns=dict_rename, inplace=True)

# get info
get_df_info(df=df_gen_xi)

N rows: 1540964; N columns: 2421
CPU times: user 1min 58s, sys: 3min 14s, total: 5min 12s
Wall time: 30.9 s


In [102]:
# preview
df_gen_xi['ApplicationDate__app'].head(5)

0    2013-10-01
1    2013-10-01
2    2013-10-01
3    2013-10-01
4    2013-10-01
Name: ApplicationDate__app, dtype: object

In [103]:
# convert to dtime
df_gen_xi['ApplicationDate__app'] = pd.to_datetime(df_gen_xi['ApplicationDate__app'])

# preview
df_gen_xi['ApplicationDate__app'].head(5)

0   2013-10-01
1   2013-10-01
2   2013-10-01
3   2013-10-01
4   2013-10-01
Name: ApplicationDate__app, dtype: datetime64[ns]

In [104]:
# check for duplicate IDs
df_gen_xi.drop_duplicates(subset=['UniqueID'], keep='last', inplace=True)
# get info
get_df_info(df=df_gen_xi)

N rows: 1540964; N columns: 2421


In [105]:
# list_to_check
list_to_check = list(df_base['uniqueid'].dropna())
# create bit field
df_gen_xi['bit_drop'] = df_gen_xi['UniqueID'].isin(list_to_check) * 1
# message
print(f'There are {np.sum(df_gen_xi["bit_drop"])} rows in Gen XI data with UniqueID found in Gen XII data')

There are 18299 rows in Gen XI data with UniqueID found in Gen XII data


In [106]:
# drop dups
df_gen_xi = df_gen_xi[df_gen_xi['bit_drop']==0].copy()
# drop bit field
df_gen_xi.drop(['bit_drop'], axis=1, inplace=True)
# get info
get_df_info(df=df_gen_xi)

N rows: 1522665; N columns: 2421


In [107]:
#pprint(list(df_gen_xi.columns))

In [108]:
# reverse match
dict_match = dict(zip(list_cols, list_cols_match))
#pprint(dict_match)

In [109]:
# rename
df_gen_xi.rename(columns=dict_match, inplace=True)
#pprint(list(df_gen_xi.columns))

In [110]:
df_gen_xi.head()

,uniqueid,target,applicationdate__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,strzipcode__app,bitapproved__app,...,us934s__tu,us935b__tu,us935c__tu,us935d__tu,us935s__tu,bcpmtstr__tu,bcpmtnum__tu,score_bankcard__tu,score_cvpropensity__tu,finscore__tu
0,133750517564321,1,2013-10-01,1337505,1756432,1,ALBUQUERQUE,New Mexico,87121,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,133750617564331,1,2013-10-01,1337506,1756433,1,ALBUQUERQUE,New Mexico,87121,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,133750617564340,1,2013-10-01,1337506,1756434,0,ALBUQUERQUE,New Mexico,87121,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,133750717564351,1,2013-10-01,1337507,1756435,1,ALLIANCE,Ohio,44601,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,133751117564401,1,2013-10-01,1337511,1756440,1,REYNOLDSBURG,Ohio,43068,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [111]:
# list of columns that are duplicated
list_cols = [
    'analyticsmatchkey__tu',
    'observationdate__tu',
    'state__tu',
    'indflag__tu',
    'zip5__tu',
]

# drop duplicate columns
df_gen_xi.drop(list_cols, axis=1, inplace=True)
# get info
get_df_info(df=df_gen_xi)

N rows: 1522665; N columns: 2409


## Concatenate ```df_base``` and ```df_gen_xi```

In [112]:
# concatenate
df_base = pd.concat([df_base, df_gen_xi], axis=0)

# save memory
del df_gen_xi

# get info
get_df_info(df=df_base)

N rows: 2100698; N columns: 2488


In [113]:
#pprint(list(df_base.columns))

In [114]:
# make sure no rows are NaN in ID, date, and target cols
df_base[['uniqueid', 'applicationdate__app', 'target']].isnull().mean()

uniqueid                0.068426
applicationdate__app    0.000000
target                  0.000000
dtype: float64

In [115]:
# make sure there are no duplicate ids
print(f'There are {df_base["uniqueid"].dropna().duplicated().sum()} duplicate IDs')

There are 0 duplicate IDs


In [116]:
# make sure there are no nan strings
df_base.replace(['NaN','nan',''], np.nan, inplace=True)

### Get the riskview score

In [117]:
%%time

str_filename = 'df_riskview_ad.csv'
str_uri = f's3://{str_project}/ad_hoc/riskview_score_ad/{str_filename}'
df_score = pd.read_csv(str_uri)

# rename
dict_rename = {
    'bigaccountid__ln': 'bigaccountid__app',
    'bigdebtorid__ln': 'bigdebtorid__app',
}
df_score.rename(columns=dict_rename, inplace=True)

# drop
df_score.drop('dtmstampcreation__ln', axis=1, inplace=True)

# show
df_score

CPU times: user 2.18 s, sys: 150 ms, total: 2.33 s
Wall time: 12.9 s


,bigaccountid__app,bigdebtorid__app,intscore__ln
0,848171,1128350,585
1,848171,1128351,555
2,848172,1128352,655
3,848173,1128353,621
4,848184,1128366,521
...,...,...,...
2612646,3763712,4846824,591
2612647,3763713,4846826,522
2612648,3763719,4846834,600
2612649,3763721,4846837,598


In [118]:
%%time

# drop
df_base.drop('intscore__ln', axis=1, inplace=True)

# join to get intscore__ln
df_base = pd.merge(
    left=df_base,
    right=df_score,
    on=['bigaccountid__app','bigdebtorid__app'],
    how='left',
    
)

# show
df_base

CPU times: user 16.3 s, sys: 14.3 s, total: 30.6 s
Wall time: 30.6 s


,bitdebtor__base,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,dtmfunded__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,...,us934d__tu,us934s__tu,us935b__tu,us935c__tu,us935d__tu,us935s__tu,score_bankcard__tu,score_cvpropensity__tu,finscore__tu,intscore__ln
0,0.0,20181103.0,20181103.0,NaN,NaN,201810.0,73140.0,2018-11-03,0.0,AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,20181103.0,NaN,20181103.0,NaN,201810.0,72817.0,2018-11-03,0.0,AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.0,20181103.0,NaN,20181103.0,NaN,201810.0,74494.0,2018-11-03,0.0,AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.0,20181030.0,NaN,20181030.0,NaN,201810.0,61357.0,2018-10-30,0.0,AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.0,20181031.0,20181031.0,NaN,NaN,201810.0,61417.0,2018-10-31,0.0,AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2100693,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,NaN,555.0,588.0
2100694,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,451.0,571.0,574.0
2100695,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,447.0,616.0,568.0
2100696,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,501.0,648.0,595.0


## Write to s3

In [119]:
%%time

# get non numeric cols and set as string
for col in df_base.columns:
    if df_base[col].dtype not in ['int64','float64']:
        df_base[col] = df_base[col].astype(str)

CPU times: user 16.4 s, sys: 2.51 s, total: 18.9 s
Wall time: 18.9 s


In [120]:
%%time

# to parquet
str_filename = 'df_raw.gzip'
str_uri = f's3://{str_project}/01_ad/01_data_prep/01_data_collection/output/{str_filename}'
df_base.to_parquet(
    path=str_uri,
    compression='gzip',
)

CPU times: user 13min 3s, sys: 4.89 s, total: 13min 8s
Wall time: 12min 27s
